In [0]:
display(spark.sql("SHOW TABLES IN main.default"))

In [0]:
from pyspark.sql.functions import lit, col
from functools import reduce

upi_tables = {
    "01": "upi_jan_2025",
    "02": "upi_feb_2025",
    "03": "upi_mar_2025",
    "04": "upi_apr_2025",
    "05": "upi_may_2025",
    "06": "upi_jun_2025",
    "07": "upi_jul_2025",
    "08": "upi_aug_2025",
    "09": "upi_sep_2025",
    "10": "upi_oct_2025",
    "11": "upi_nov_2025",
    "12": "upi_dec_2025"
}

# Standardized column names (8 columns in each table)
standard_columns = [
    "sr_no",
    "upi_remitter_banks",
    "total_volume_in_mn",
    "approved_pct",
    "bd_pct",
    "td_pct",
    "total_debit_reversal_count_in_mn",
    "debit_reversal_success_pct"
]

dfs = []
for month, table_name in upi_tables.items():
    # Read table
    df = spark.read.table(f"main.default.{table_name}")
    
    # Get current column names (they vary by month)
    current_cols = df.columns
    
    # Rename to standard names
    for i, std_name in enumerate(standard_columns):
        df = df.withColumnRenamed(current_cols[i], std_name)
    
    # Filter out the header row (where sr_no = 'Sr. No.')
    df = df.filter(col("sr_no") != "Sr. No.")
    
    # Add source_month column
    df = df.withColumn("source_month", lit(month))
    
    dfs.append(df)

# Union all dataframes (now they have matching column names)
upi_all = reduce(lambda a, b: a.union(b), dfs)

display(upi_all)

In [0]:
# Write to Delta table (columns are already clean)
# Use option to allow schema overwrite
upi_all.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_upi_combined")

In [0]:
display(spark.sql("SELECT source_month, COUNT(*) as row_count FROM bronze_upi_combined GROUP BY source_month ORDER BY source_month"))

In [0]:
display(spark.sql("SELECT * FROM bronze_upi_combined WHERE source_month = '01' LIMIT 60"))

In [0]:
for month, table_name in upi_tables.items():
    df = spark.read.table(f"main.default.{table_name}")
    print(month, table_name, "→", df.count(), "rows,", len(df.columns), "columns")

In [0]:
from pyspark.sql.functions import lit
from functools import reduce

column_names = [
    "sr_no", "bank_name", "total_volume_mn", "approved_pct",
    "bd_pct", "td_pct", "reversal_count_mn", "reversal_success_pct"
]

upi_tables = {
    "01": "upi_jan_2025", "02": "upi_feb_2025", "03": "upi_mar_2025",
    "04": "upi_apr_2025", "05": "upi_may_2025", "06": "upi_jun_2025",
    "07": "upi_jul_2025", "08": "upi_aug_2025", "09": "upi_sep_2025",
    "10": "upi_oct_2025", "11": "upi_nov_2025", "12": "upi_dec_2025"
}

dfs = []
for month, table_name in upi_tables.items():
    df = spark.read.table(f"main.default.{table_name}")   # use the full path, like your working cell above
    df = df.toDF(*column_names)
    df = df.filter(df.sr_no != "Sr. No.")
    df = df.withColumn("source_month", lit(month))
    dfs.append(df)

upi_all = reduce(lambda a, b: a.unionByName(b), dfs)
display(upi_all)

In [0]:
from pyspark.sql.functions import expr

upi_all_clean = upi_all.filter(expr("try_cast(sr_no as int) IS NOT NULL"))

In [0]:
display(upi_all_clean.groupBy("source_month").count().orderBy("source_month"))

In [0]:
upi_all_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_upi_combined")

In [0]:
display(spark.sql("SELECT source_month, COUNT(*) FROM bronze_upi_combined GROUP BY source_month ORDER BY source_month"))

In [0]:
display(spark.sql("SHOW TABLES IN main.default"))

In [0]:
display(spark.sql("SELECT * FROM bronze_upi_combined"))

In [0]:
display(spark.sql("SHOW TABLES IN main.default"))

In [0]:
npa_raw = spark.read.table("main.default.npa_movement_fy_2425")  # confirm exact name first
display(npa_raw)

advances_raw = spark.read.table("main.default.advances_fy_2425")
display(advances_raw)

In [0]:
from pyspark.sql.functions import col, lit

npa_columns = [
    "year", "bank_name", "npa_opening_cr", "npa_addition_cr",
    "npa_reduction_cr", "npa_writeoff_cr", "npa_closing_cr",
    "net_npa_opening_cr", "net_npa_closing_cr"
]

npa_raw = spark.read.table("main.default.npa_movement_2025").toDF(*npa_columns)

# Drop the 2 junk header rows — they're the only rows where bank_name is empty
npa_clean = npa_raw.filter(col("bank_name").isNotNull())

# Since we already know this whole sheet is just FY2024-25, just set it directly
# (simpler and safer than trying to "fill down" the blank year cells)
npa_clean = npa_clean.withColumn("year", lit("2025"))

display(npa_clean)

In [0]:
from pyspark.sql.functions import col, lit

npa_columns = [
    "blank_col", "year_raw", "bank_name", "npa_opening_cr", "npa_addition_cr",
    "npa_reduction_cr", "npa_writeoff_cr", "npa_closing_cr", "net_npa_opening_cr"
]

npa_raw = spark.read.table("main.default.npa_movement_2025").toDF(*npa_columns)
npa_raw = npa_raw.drop("blank_col", "year_raw")
npa_raw = npa_raw.withColumn("year", lit("2025"))
npa_clean = npa_raw.filter(col("bank_name").isNotNull())

display(npa_clean)

In [0]:
display(npa_clean.select("bank_name").distinct().orderBy("bank_name"))

In [0]:
group_rows = ["PUBLIC SECTOR BANKS", "PRIVATE SECTOR BANKS", "FOREIGN BANKS"]

# Your main table — individual banks only, this is what joins to UPI data
npa_individual = npa_clean.filter(~npa_clean.bank_name.isin(group_rows))

# A separate small table — bank-group totals, kept aside as a bonus/optional view
npa_group_totals = npa_clean.filter(npa_clean.bank_name.isin(group_rows))

display(npa_individual)

In [0]:
npa_individual.write.format("delta").mode("overwrite").saveAsTable("bronze_npa")

In [0]:
advances_raw = spark.read.table("main.default.advances_raw")
display(advances_raw.columns)

In [0]:
display(spark.sql("SELECT * FROM main.default.advances_raw LIMIT 15"))

In [0]:
from pyspark.sql.functions import col, lit

advances_clean = advances_raw.select(
    col("_c2").alias("bank_name"),
    col("_c20").alias("advances_cr")
)

# Drop the leftover header row and any blank rows
advances_clean = advances_clean.filter(
    col("bank_name").isNotNull() & (col("bank_name") != "Banks")
)

# Remove group/subtotal rows — same category names we found in the NPA table
group_rows = ["PUBLIC SECTOR BANKS", "PRIVATE SECTOR BANKS", "FOREIGN BANKS"]
advances_individual = advances_clean.filter(~col("bank_name").isin(group_rows))

# We already know this file is FY2024-25
advances_individual = advances_individual.withColumn("year", lit("2025"))

display(advances_individual)

In [0]:
advances_individual.write.format("delta").mode("overwrite").saveAsTable("bronze_advances")

In [0]:
display(spark.sql("SELECT * FROM bronze_upi_combined"))

In [0]:
display(spark.sql("SELECT * FROM bronze_npa"))

In [0]:
display(spark.sql("SELECT * FROM bronze_advances"))